In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.cfpb_complaints_bronze")

missing = df.select([
    F.sum(
        F.when(F.col(f"`{c}`").isNull() | (F.trim(F.col(f"`{c}`")) == "null"), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

missing.show(vertical=True)

-RECORD 0----------------------
 _c0                  | 0      
 product_5            | 0      
 narrative            | 0      
 Product              | 0      
 Date_received        | 0      
 Sub-product          | 52206  
 Issue                | 0      
 Sub-issue            | 230559 
 Company              | 0      
 State                | 7344   
 Timely_response?     | 0      
 _ingestion_timestamp | 0      
 _source_file         | 0      
 _source_system       | 0      



+--------------------------+-------+
|product_5                 |count  |
+--------------------------+-------+
|Credit Reporting          |1205275|
|Debt Collection           |266842 |
|Loans                     |228599 |
|Credit Card Services      |163710 |
|Bank Accounts and Services|158640 |
+--------------------------+-------+



In [0]:
from pyspark.sql import functions as F
df = spark.table("workspace.default.cfpb_complaints_bronze")
df.groupBy("`Timely_response?`").count().show()

+----------------+-------+
|Timely_response?|  count|
+----------------+-------+
|             Yes|1996762|
|              No|  26304|
+----------------+-------+



In [0]:
from pyspark.sql import functions as F
df = spark.table("workspace.default.cfpb_complaints_bronze")
df.select(F.min("Date_received"), F.max("Date_received")).show()

+------------------+------------------+
|min(Date_received)|max(Date_received)|
+------------------+------------------+
|        2015-03-19|        2024-07-31|
+------------------+------------------+



In [0]:
df = df.fillna({
    "sub_product": "Not specified",
    "sub_issue": "Not specified",
    "state": "Unknown"
})

In [0]:
total = df.count()
unique_ids = df.select("complaint_id").distinct().count()
print(total, unique_ids)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4704975761107855>, line 1
----> 1 total = df.count()
      2 unique_ids = df.select("complaint_id").distinct().count()
      3 print(total, unique_ids)

File <command-4704975761107854>, line 1
----> 1 df = df.fillna({
      2     "sub_product": "Not specified",
      3     "sub_issue": "Not specified",
      4     "state": "Unknown"
      5 })

File <command-4704975761107853>, line 2
      1 from pyspark.sql import functions as F
----> 2 df = spark.table("workspace.default.cfpb_complaints_bronze")
      3 df.select(F.min("Date_received"), F.max("Date_received")).show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `sub_product` cannot be resolved. Did you mean one of the following? [`_c0`, `product_5`, `narrative`, `Product`, `Date_received`, `Sub-product`, `Iss

In [0]:
from pyspark.sql import functions as F

# 1. Read Bronze
df = spark.table("workspace.default.cfpb_complaints_bronze")

# 2. Rename columns
df = (
    df
    .withColumnRenamed("_c0", "complaint_id")
    .withColumnRenamed("Date_received", "date_received")
    .withColumnRenamed("Sub-product", "sub_product")
    .withColumnRenamed("Sub-issue", "sub_issue")
    .withColumnRenamed("Timely_response?", "timely_response")
    .withColumnRenamed("Product", "product")
    .withColumnRenamed("Issue", "issue")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("State", "state")
)

# 3. Turn the text "null" into real empty values
text_cols = ["product_5", "narrative", "product", "sub_product",
             "issue", "sub_issue", "company", "state", "timely_response"]

for c in text_cols:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "null", None).otherwise(F.trim(F.col(c)))
    )

# 4. Text date -> real date
df = df.withColumn("date_received", F.to_date("date_received", "yyyy-MM-dd"))

# 5. Fill empty values
df = df.fillna({
    "sub_product": "Not specified",
    "sub_issue": "Not specified",
    "state": "Unknown"
})

# 6. Check
print("Rows:", df.count())
print("Unique IDs:", df.select("complaint_id").distinct().count())
print("Rows with no date:", df.filter(F.col("date_received").isNull()).count())
df.printSchema()

Rows: 2023066
Unique IDs: 2023066
Rows with no date: 0
root
 |-- complaint_id: string (nullable = true)
 |-- product_5: string (nullable = true)
 |-- narrative: string (nullable = true)
 |-- product: string (nullable = true)
 |-- date_received: date (nullable = true)
 |-- sub_product: string (nullable = false)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = false)
 |-- company: string (nullable = true)
 |-- state: string (nullable = false)
 |-- timely_response: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)



In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.cfpb_complaints_silver")
)

print(spark.table("workspace.default.cfpb_complaints_silver").count())

2023066
